# Fine-tuning Llama-PLLuM-8B-chat — QLoRA

- **Model:** `CYFRAGOVPL/Llama-PLLuM-8B-chat`
- **License:** Llama 3.1 (open, commercial use permitted)
- **Method:** QLoRA (4-bit) via Unsloth
- **Kaggle**: `T4 GPU x 2`
- **Training time:** ~5 min (1 epoch, small dataset)

### Kaggle → Settings:
- `Kaggle` → `Settings` → `Accelerator` → **T4 GPU x 2**

## 1. Installation

In [1]:
%%capture
!pip install unsloth
!pip install trl accelerate peft datasets bitsandbytes

## 2. GPU check

In [ ]:
!nvidia-smi

## 3. Loading the model + QLoRA

> **Note:** Downloading the 8B model will take ~2-3 min. VRAM after loading: approx. 6-7 GB.

In [ ]:
import unsloth
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="CYFRAGOVPL/Llama-PLLuM-8B-chat",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

# LoRA adapters - target modules identical to those for Llama 3.1
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    # lora_dropout=0.05,
    lora_dropout=0.00,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"VRAM zajęte: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## 4. Dataset

Example data in the format `{instruction, input, output}`

> ** Important:** PLLuM uses the **Llama 3** format (`<|start_header_id|>`), NOT ChatML (`<|im_start|>`).


In [ ]:
from datasets import Dataset

# ===== DATA =====
data = [
    {
        "instruction": "Odpowiedz krótko po polsku.",
        "input": "Co to jest machine learning?",
        "output": "Machine learning to dziedzina AI, w której modele uczą się na danych bez jawnego programowania."
    },
    {
        "instruction": "Odpowiedz krótko po polsku.",
        "input": "Co to jest fine-tuning?",
        "output": "Fine-tuning to dostosowanie pretrenowanego modelu do konkretnego zadania poprzez dalsze trenowanie na małym zbiorze danych."
    },
    {
        "instruction": "Odpowiedz krótko po polsku.",
        "input": "Co to jest LoRA?",
        "output": "LoRA (Low-Rank Adaptation) to metoda fine-tuningu, która trenuje tylko małe adaptery zamiast całego modelu — oszczędza VRAM i czas."
    },
    {
        "instruction": "Odpowiedz krótko po polsku.",
        "input": "Co to jest tokenizer?",
        "output": "Tokenizer to algorytm zamieniający tekst na liczby (tokeny) zrozumiałe dla modelu językowego."
    },
    {
        "instruction": "Odpowiedz krótko po polsku.",
        "input": "Co to jest GPU?",
        "output": "GPU to procesor graficzny — ma tysiące małych rdzeni, idealnych do równoległych obliczeń macierzowych potrzebnych w AI."
    },
]
# ===== END OF DATA =====

# Chat template format - LLAMA 3 (not ChatML!)
# PLLuM uses: <|begin_of_text|><|start_header_id|>role<|end_header_id|>\n\ntcontent<|eot_id|>
def format_prompt(example):
    text = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n\n"
        f"{example['instruction']}<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        f"{example['input']}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{example['output']}<|eot_id|>"
    )
    return {"text": text}

dataset = Dataset.from_list(data).map(format_prompt)
print(f"Examples in the dataset: {len(dataset)}")
print("\nExample:")
print(dataset[0]["text"])

## 5. Training

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig( 
        dataset_text_field="text",
        max_seq_length=512, 
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=1,
        output_dir="./output",
        optim="adamw_8bit",
        save_strategy="no",
        report_to="none",
    ),
)

print("Start training...")
trainer.train()
print("✅ Training finished!")

## 6. Test of the model after fine-tuning

In [ ]:
FastLanguageModel.for_inference(model)

def ask(question, instruction="Odpowiedz krótko po polsku."):
    prompt = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n\n"
        f"{instruction}<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        f"{question}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in response:
        return response.split("assistant")[-1].strip()
    return response.strip()

# Test
print(ask("Co to jest LoRA?"))
print("---")
print(ask("Wyjaśnij czym jest PLLuM."))